In [1]:
import os
import gc

import numpy as np
import pandas as pd
import seaborn as sns
import missingno as msno
sns.set(style='darkgrid')
import matplotlib.pyplot as plt
from pathlib import Path
import joblib

print("Working directory:", Path.cwd())

data_list = [pd.read_csv(csv) for csv in Path("../data/").iterdir()]
data = pd.concat(data_list)

# Deleting dataframes after concating to save memory
for d in data_list: del d
gc.collect()

# Renaming the columns by removing leading/trailing whitespace
col_names = {col: col.strip() for col in data.columns}
data.rename(columns = col_names, inplace = True)
data.columns

print("done entering data")

Working directory: /home/weluv/CMPE-272/src
done entering data


In [3]:
for col in data.select_dtypes(include='float').columns:
    if data[col].dropna().mod(1).eq(0).all():
        data[col] = data[col].astype('int64')

data.drop(columns=['[ZoneTransfer]'], errors='ignore', inplace=True)
data['Label'] = data['Label'].astype('object')

data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2830743 entries, 0 to 288601
Data columns (total 79 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Destination Port             int64  
 1   Flow Duration                int64  
 2   Total Fwd Packets            int64  
 3   Total Backward Packets       int64  
 4   Total Length of Fwd Packets  int64  
 5   Total Length of Bwd Packets  int64  
 6   Fwd Packet Length Max        int64  
 7   Fwd Packet Length Min        int64  
 8   Fwd Packet Length Mean       float64
 9   Fwd Packet Length Std        float64
 10  Bwd Packet Length Max        int64  
 11  Bwd Packet Length Min        int64  
 12  Bwd Packet Length Mean       float64
 13  Bwd Packet Length Std        float64
 14  Flow Bytes/s                 float64
 15  Flow Packets/s               float64
 16  Flow IAT Mean                float64
 17  Flow IAT Std                 float64
 18  Flow IAT Max                 int64  
 19  Flow I

In [4]:
pd.options.display.max_columns = 80
data

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,49188,4,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,3000000.0,5.000000e+05,4.000000e+00,0.000000,4,4,4,4.000000e+00,0.000000,4,4,0,0.000,0.000000,0,0,0,0,0,0,40,0,5.000000e+05,0.0,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,0,9.0,6.0,0.0,40,0,0,0,0,0,0,2,12,0,0,329,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,49188,1,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,12000000.0,2.000000e+06,1.000000e+00,0.000000,1,1,1,1.000000e+00,0.000000,1,1,0,0.000,0.000000,0,0,0,0,0,0,40,0,2.000000e+06,0.0,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,0,9.0,6.0,0.0,40,0,0,0,0,0,0,2,12,0,0,329,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,49188,1,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,12000000.0,2.000000e+06,1.000000e+00,0.000000,1,1,1,1.000000e+00,0.000000,1,1,0,0.000,0.000000,0,0,0,0,0,0,40,0,2.000000e+06,0.0,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,0,9.0,6.0,0.0,40,0,0,0,0,0,0,2,12,0,0,329,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,49188,1,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,12000000.0,2.000000e+06,1.000000e+00,0.000000,1,1,1,1.000000e+00,0.000000,1,1,0,0.000,0.000000,0,0,0,0,0,0,40,0,2.000000e+06,0.0,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,0,9.0,6.0,0.0,40,0,0,0,0,0,0,2,12,0,0,329,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,49486,3,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,4000000.0,6.666667e+05,3.000000e+00,0.000000,3,3,3,3.000000e+00,0.000000,3,3,0,0.000,0.000000,0,0,0,0,0,0,40,0,6.666667e+05,0.0,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,0,9.0,6.0,0.0,40,0,0,0,0,0,0,2,12,0,0,245,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
288597,80,590930,2,0,0,0,0,0,0.0,0.0,0,0,0.0,0.0,0.0,3.384496e+00,5.909300e+05,0.000000,590930,590930,590930,5.909300e+05,0.000000,590930,590930,0,0.000,0.000000,0,0,0,0,0,0,64,0,3.384496e+00,0.0,0,0,0.0,0.0,0.0,0,0,0,0,1,1,0,0,0,0.0,0.0,0.0,64,0,0,0,0,0,0,2,0,0,0,238,-1,0,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
288598,80,1187988,2,0,0,0,0,0,0.0,0.0,0,0,0.0,0.0,0.0,1.683519e+00,1.187988e+06,0.000000,1187988,1187988,1187988,1.187988e+06,0.000000,1187988,1187988,0,0.000,0.000000,0,0,0,0,0,0,64,0,1.683519e+00,0.0,0,0,0.0,0.0,0.0,0,0,0,0,1,1,0,0,0,0.0,0.0,0.0,64,0,0,0,0,0,0,2,0,0,0,238,-1,0,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
288599,80,10,1,9,6,54,6,6,6.0,0.0,6,6,6.0,0.0,6000000.0,1.000000e+06,1.111111e+00,0.781736,3,0,0,0.000000e+00,0.000000,0,0,7,0.875,0.353553,1,0,0,0,0,0,20,180,1.000000e+05,900000.0,6,6,6.0,0.0,0.0,1,0,0,0,1,0,0,0,9,6.6,6.0,6.0,20,0,0,0,0,0,0,1,6,9,54,0,0,0,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
288600,138,19,10,0,2370,0,237,237

In [5]:
# data cleaning

data.drop_duplicates(inplace = True)
data.replace([np.inf, -np.inf], np.nan, inplace = True)

# Filling missing values with median
med_flow_bytes = data['Flow Bytes/s'].median()
med_flow_packets = data['Flow Packets/s'].median()
data['Flow Bytes/s'].fillna(med_flow_bytes, inplace = True)
data['Flow Packets/s'].fillna(med_flow_packets, inplace = True)

print("done")
data.info()

done
<class 'pandas.core.frame.DataFrame'>
Index: 2522362 entries, 0 to 288601
Data columns (total 79 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Destination Port             int64  
 1   Flow Duration                int64  
 2   Total Fwd Packets            int64  
 3   Total Backward Packets       int64  
 4   Total Length of Fwd Packets  int64  
 5   Total Length of Bwd Packets  int64  
 6   Fwd Packet Length Max        int64  
 7   Fwd Packet Length Min        int64  
 8   Fwd Packet Length Mean       float64
 9   Fwd Packet Length Std        float64
 10  Bwd Packet Length Max        int64  
 11  Bwd Packet Length Min        int64  
 12  Bwd Packet Length Mean       float64
 13  Bwd Packet Length Std        float64
 14  Flow Bytes/s                 float64
 15  Flow Packets/s               float64
 16  Flow IAT Mean                float64
 17  Flow IAT Std                 float64
 18  Flow IAT Max                 int64  
 19  F

/tmp/ipykernel_18767/3742571967.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Flow Bytes/s'].fillna(med_flow_bytes, inplace = True)
/tmp/ipykernel_18767/3742571967.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplac

In [6]:
# Creating a dictionary that maps each label to its attack type
attack_map = {
    'BENIGN': 'BENIGN',
    'DDoS': 'DDoS',
    'DoS Hulk': 'DoS',
    'DoS GoldenEye': 'DoS',
    'DoS slowloris': 'DoS',
    'DoS Slowhttptest': 'DoS',
    'PortScan': 'Port Scan',
    'FTP-Patator': 'Brute Force',
    'SSH-Patator': 'Brute Force',
    'Bot': 'Bot',
    'Web Attack � Brute Force': 'Web Attack',
    'Web Attack � XSS': 'Web Attack',
    'Web Attack � Sql Injection': 'Web Attack',
    'Infiltration': 'Infiltration',
    'Heartbleed': 'Heartbleed'
}

# Creating a new column 'Attack Type' in the DataFrame based on the attack_map dictionary
if 'Label' in data.columns:
    data['Attack Type'] = data['Label'].map(attack_map)

    data.drop('Label', axis = 1, inplace = True)

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
data['Attack Number'] = le.fit_transform(data['Attack Type'])

print("done mapping data")

done mapping data


In [7]:
# data processing

for col in data.columns:
    col_type = data[col].dtype
    if col_type != object:
        c_min = data[col].min()
        c_max = data[col].max()
        # Downcasting float64 to float32
        if str(col_type).find('float') >= 0 and c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
            data[col] = data[col].astype(np.float32)

        # Downcasting int64 to int32
        elif str(col_type).find('int') >= 0 and c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
            data[col] = data[col].astype(np.int32)
            
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2522362 entries, 0 to 288601
Data columns (total 80 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Destination Port             int32  
 1   Flow Duration                int32  
 2   Total Fwd Packets            int32  
 3   Total Backward Packets       int32  
 4   Total Length of Fwd Packets  int32  
 5   Total Length of Bwd Packets  int32  
 6   Fwd Packet Length Max        int32  
 7   Fwd Packet Length Min        int32  
 8   Fwd Packet Length Mean       float32
 9   Fwd Packet Length Std        float32
 10  Bwd Packet Length Max        int32  
 11  Bwd Packet Length Min        int32  
 12  Bwd Packet Length Mean       float32
 13  Bwd Packet Length Std        float32
 14  Flow Bytes/s                 float32
 15  Flow Packets/s               float32
 16  Flow IAT Mean                float32
 17  Flow IAT Std                 float32
 18  Flow IAT Max                 int32  
 19  Flow I

In [8]:
# Dropping columns with only one unique value
num_unique = data.nunique()
one_variable = num_unique[num_unique == 1]
not_one_variable = num_unique[num_unique > 1].index

dropped_cols = one_variable.index
data = data[not_one_variable]

print('Dropped columns:')
dropped_cols

Dropped columns:


Index(['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk',
       'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk',
       'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate'],
      dtype='object')

In [9]:
# Columns after removing non variant columns
data.columns

Index(['Destination Port', 'Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Total Length of Fwd Packets',
       'Total Length of Bwd Packets', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd URG Flags',
       'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s',
       'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length',
       'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance',
       'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count',

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Config
CV_FOLDS = 5
SEED = 42

# Data prep. 
# replace y with the commented out stuff if we want multi-class
"""
y = le.fit_transform(data["Attack Type"].astype(str))
""" 
y = (data["Attack Type"].astype(str).str.upper() != "BENIGN").astype(np.int8).to_numpy()
X = data.drop(columns=["Attack Type", "Attack Number"]).to_numpy(dtype=np.float32)
feature_names = [c for c in data.columns if c not in ("Attack Type", "Attack Number")]
attack_type = data["Attack Type"].to_numpy()   # kept for per-family breakdown
# del data
gc.collect()
 
X_train, X_test, y_train, y_test, _, type_test = train_test_split(
    X, y, attack_type, test_size=0.2, stratify=y, random_state=SEED
)
print(type(X_train))
del X, y
gc.collect()
print(f"Train {X_train.shape}, test {X_test.shape}, attack rate {y_train.mean():.3f}")

# Model: no scaler / PCA / SMOTE needed
model = HistGradientBoostingClassifier(
    max_iter=75,
    learning_rate=0.1,
    class_weight="balanced",   # handles imbalance without synthetic samples
    early_stopping=True,       # internal validation split stops training when it plateaus
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=SEED,
    verbose=1
)
 
# Stratified k-fold cross-validation
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
res = cross_validate(
    model, X_train, y_train, cv=cv,
    scoring=["accuracy", "f1", "precision", "recall"],
    n_jobs=1,   # each fit already uses all cores; parallel folds would multiply memory
)
for m in ["accuracy", "f1", "precision", "recall"]:
    s = res[f"test_{m}"]
    print(f"CV {m:<9}: {np.round(s, 4)}  mean={s.mean():.4f} (+/- {s.std():.4f})")
 
# Final model on the full training set, scored on the untouched test set
model.fit(X_train, y_train)
print(f"\nStopped at {model.n_iter_} iterations")

# dump columns for use later
joblib.dump(model, "hgb_model.joblib")
joblib.dump(le, "label_encoder.joblib")
joblib.dump(feature_names, "feature_columns.joblib")
 
y_pred = model.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Test F1:       {f1_score(y_test, y_pred):.4f}")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Benign", "Attack"], digits=4))

# Recall per attack family (shows which attacks are hardest to catch)
print("\nDetection rate by original attack type (test set):")
for t in np.unique(type_test):
    mask = type_test == t
    if str(t).upper() == "BENIGN":
        print(f"  {t:<30} n={mask.sum():>8}  correctly benign: {(y_pred[mask] == 0).mean():.4f}")
    else:
        print(f"  {t:<30} n={mask.sum():>8}  detected:         {(y_pred[mask] == 1).mean():.4f}")

<class 'numpy.ndarray'>
Train (2017889, 70), test (504473, 70), attack rate 0.169
Binning 0.814 GB of training data: 5.007 s
Binning 0.090 GB of validation data: 0.296 s
Fitting gradient boosted rounds:
Fit 75 trees in 52.283 s, (2325 total leaves)
Time spent computing histograms: 33.697s
Time spent finding best splits:  0.178s
Time spent applying splits:      1.478s
Time spent predicting:           0.386s
Binning 0.814 GB of training data: 3.898 s
Binning 0.090 GB of validation data: 0.284 s
Fitting gradient boosted rounds:
Fit 75 trees in 41.105 s, (2325 total leaves)
Time spent computing histograms: 33.942s
Time spent finding best splits:  0.169s
Time spent applying splits:      1.482s
Time spent predicting:           5.507s
Binning 0.814 GB of training data: 3.419 s
Binning 0.090 GB of validation data: 0.293 s
Fitting gradient boosted rounds:
